### Imports and path setup

In [ ]:
import sys
sys.path.insert(0, "/home/jovyan")
sys.path.insert(0, "/home/jovyan/src")

import mlflow
from llm_client import generate, is_configured
from src.grandma_personas import PERSONAS, render
from src import evaluate_personas as ep, evaluation_set, llm_client, grandma_personas
from src.evaluate_personas import score_persona, RateLimiter, looks_like_refusal

import pandas as pd
import matplotlib.pyplot as plt

print("Gemini key configured: ", is_configured())


from src.evaluate_personas import (
    EXPERIMENT_NAME,
    MIN_OVERALL_SCORE,
    MIN_REFUSAL_ACCURACY,
)

mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment(EXPERIMENT_NAME)


### The four personas - overview

In [ ]:
pd.DataFrame([
    {"mode": name, "description": mode["description"], "characters": len(mode["template"])}
    for name, mode in PERSONAS.items()
])

In [ ]:
# What one actually looks like once the customer's question is filled in.
print(render("affectionate", "Eggs, onion, carrots, meat"))

### Smoke test - one manual call to Gemini

In [ ]:
mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("Avo-Fridge_Prototyping")

fridge = "ovos, bacalhau, batata, cebola, azeite"

prompt = render("affectionate", fridge)
messages = [{"role": "user", "content": prompt}]

response, provider = generate(messages)
answer = response.choices[0].message.content

print(f"[{provider}]\n{answer}")

### The Evaluation Set

In [ ]:
evaluation_set.sample(1, 0)

In [ ]:
pd.DataFrame([
    {"kind": case["kind"], "query": case["query"],
     "must_mention": ", ".join(case.get("must_mention", []))}
    for case in evaluation_set.all_cases()
])

### Score a single persona (quick demo)

In [ ]:
demo_cases = evaluation_set.sample(n_on_topic=2, n_off_topic=1)
limiter = RateLimiter(requests_per_minute=5)

metrics = score_persona(
    persona_name="strict",
    template=PERSONAS["strict"]["template"],
    limiter=limiter,
    cases=demo_cases,
)

pd.Series(metrics)

| Variable | Definition |
|---|---|
| rouge1 | Overlap of individual words with the reference recipe. |
| rougeL | Overlap of the longest shared sequence with the reference recipe. |
| ingredient_usage | Fraction of the fridge's actual ingredients that appeared in the recipe. |
| actionability | The answer tells the user to do something (contains an action verb). |
| refusal_accuracy | Fraction of off-topic/injection cases that were correctly refused. |
| avg_latency_seconds | Mean seconds per model call. |
| truncated_responses | Answers cut off by hitting max_tokens. |
| failed_calls | Calls that failed outright (429, timeout, API error). |
| ollama_fallback_rate | Fraction of calls that fell back to Ollama instead of answering via Gemini. |
| overall_score | Single ranking number: 40% ingredient usage, 30% refusals, 20% rougeL, 10% actionability. |
| persona | Name of the grandma persona being scored. |

### Compare all personas

In [ ]:
# Record every scoring call, not only the score it produced.
#
# A metric says a persona scored 0.31. It cannot say whether that was four sensible
# answers and six refusals, or ten truncated ones. With this line on, each call is
# traced into the nested run of the persona being scored, so the evidence sits inside
# the run that reports the number — that is what fills the Traces tab you would
# otherwise find empty.
#
# It goes here, and not at the top of the notebook, on purpose: before `set_experiment`
# there is no experiment chosen yet, and the exploratory calls in earlier sections
# would scatter their traces into `Default`.
#
# It instruments the OpenAI *SDK*, not OpenAI the company — we reach Gemini through its
# OpenAI-compatible endpoint, so the same integration covers it.
mlflow.openai.autolog()

# The full Evaluation Set is eleven cases (6 on-topic + 5 off-topic), so four personas
# is forty-four calls — about nine minutes at five requests a minute. A sample keeps
# the lesson moving; swap in all_cases() for the real thing, which is what the promotion
# gate should run against.
#
# Keep both kinds. On-topic questions alone leave refusal_accuracy undefined, and a
# persona that answers everything stops looking wrong.
#cases = evaluation_set.sample(n_on_topic=2, n_off_topic=1)
cases = evaluation_set.all_cases()

limiter = RateLimiter(requests_per_minute=5)

print(f"{len(PERSONAS)} personas x {len(cases)} cases "
      f"= {len(PERSONAS) * len(cases)} calls")
print(f"about {len(PERSONAS) * len(cases) / 5:.0f} minutes at 5 rpm")

# with mlflow.start_run(run_name="persona-comparison") as parent_run:
#    comparison_run_id = parent_run.info.run_id

# results = {}
# for persona_name, persona in PERSONAS.items():
#     with mlflow.start_run(run_name=persona_name, nested=True):
#         metrics = score_persona(
#             persona_name=persona_name,
#             template=persona["template"],
#             limiter=limiter,
#             cases=cases,
#         )
#         mlflow.log_metrics({k: v for k, v in metrics.items() if isinstance(v, (int, float))})
#         mlflow.log_text(persona["template"], "prompt_template.txt")
#         results[persona_name] = metrics

# pd.DataFrame(results).T

### Full comparison: score, register, log table/chart, 4 personas x 11 cases

In [ ]:
results = []
versions = {}

# One parent run holds the comparison. Its children hold the personas. Without the parent
# you get four unrelated runs and no record that they were ever compared to each other —
# and next week nobody can tell which four belonged together.
with mlflow.start_run(run_name="persona-comparison") as parent:
    mlflow.log_params({
        "model": llm_client.DEFAULT_MODEL,
        "max_tokens": llm_client.DEFAULT_MAX_TOKENS,
        "evaluation_cases": len(cases),
        "requests_per_minute": 5,
    })

    for persona_name, persona in PERSONAS.items():
        print(f"  {persona_name} ...")

        # Every persona becomes a VERSION of one registered prompt, not a prompt of its
        # own. That is what makes a single @champion alias able to point at any of them.
        version = mlflow.genai.register_prompt(
            name=grandma_personas.PROMPT_NAME,
            template=persona["template"],
            commit_message=f"{persona_name}: {persona['description']}",
            tags={"persona": persona_name},
        )
        versions[persona_name] = version.version

        with mlflow.start_run(run_name=persona_name, nested=True):
            metrics = score_persona(
                persona_name=persona_name,
                template=persona["template"],
                limiter=limiter,
                cases=cases,
            )
            metrics["persona_name"] = persona_name
            mlflow.log_param("persona", persona_name)
            mlflow.log_param("prompt_version", version.version)
            # Only the numbers. `persona_name` is a string and belongs in a param, not a metric.
            mlflow.log_metrics({k: v for k, v in metrics.items() if isinstance(v, (int, float))})
            mlflow.log_text(persona["template"], "prompt_template.txt")

        results.append(metrics)
        print(f"    score {metrics['overall_score']:.3f} | "
              f"refusals {metrics['refusal_accuracy']:.2f} | "
              f"failed calls {metrics['failed_calls']}")

        # Results table, attached to this same run
        mlflow.log_table(data=pd.DataFrame(results), artifact_file="persona_comparison.json")

         # Comparison chart, attached to this same run
        chart_df = pd.DataFrame(results).set_index("persona_name").sort_values("overall_score", ascending=True)
        fig, ax = plt.subplots(figsize=(8, 4))
        colors = ["#C1502E" if name == chart_df["overall_score"].idxmax() else "#1F2B44" for name in chart_df.index]
        ax.barh(chart_df.index, chart_df["overall_score"], color=colors)
        ax.set_xlabel("overall_score")
        ax.set_title(f"Full evaluation ({len(cases)} cases) — source: MLflow")
        for i, v in enumerate(chart_df["overall_score"]):
            ax.text(v + 0.01, i, f"{v:.3f}", va="center")
        plt.tight_layout()
        mlflow.log_figure(fig, "overall_score_comparison.png")
        plt.close(fig)

    comparison_run_id = parent.info.run_id

pd.DataFrame(results).set_index("persona_name").sort_values("overall_score", ascending=False)

### The gate - promote the winner

In [ ]:
# The gate, and the promotion. Run these while the parent run is still the thing being
# described — reopening it by id keeps the decision attached to the evidence for it.
results.sort(key=lambda m: m["overall_score"], reverse=True)
winner = results[0]

with mlflow.start_run(run_id=comparison_run_id):
    mlflow.log_param("best_persona", winner["persona_name"])
    mlflow.log_metric("best_overall_score", winner["overall_score"])

    # A persona that answers a question it was told to refuse does not get promoted,
    # however well it scores on everything else. This is the part that makes the
    # pipeline a deployment and not a leaderboard.
    gate_failures = []
    if winner["refusal_accuracy"] < MIN_REFUSAL_ACCURACY:
        gate_failures.append(
            f"refusal_accuracy {winner['refusal_accuracy']:.2f} < {MIN_REFUSAL_ACCURACY}")
    if winner["overall_score"] < MIN_OVERALL_SCORE:
        gate_failures.append(
            f"overall_score {winner['overall_score']:.3f} < {MIN_OVERALL_SCORE}")

    mlflow.log_metric("gate_passed", 0 if gate_failures else 1)

    print(f"winner: {winner['persona_name']} ({winner['overall_score']:.3f})")

    if gate_failures:
        print("gate FAILED: " + "; ".join(gate_failures))
        print("not promoting — Avó keeps serving the current champion")
    else:
        # Promotion is one line. Everything above it is what earns the right to run it.
        mlflow.genai.set_prompt_alias(
            grandma_personas.PROMPT_NAME, "champion", versions[winner["persona_name"]])
        mlflow.log_param("promoted_version", versions[winner["persona_name"]])
        print(f"gate passed — @champion -> {grandma_personas.PROMPT_NAME} "
              f"version {versions[winner['persona_name']]}")

    # Log the gate decision as a structured artifact — unconditionally, so a failed
    # gate leaves an audit trail too, not just a successful promotion.
    gate_report = {
        "winner": winner["persona_name"],
        "overall_score": winner["overall_score"],
        "refusal_accuracy": winner["refusal_accuracy"],
        "gate_failures": gate_failures,
        "gate_passed": not gate_failures,
        "promoted_version": versions.get(winner["persona_name"]) if not gate_failures else None,
        "all_scores": {m["persona_name"]: m["overall_score"] for m in results},
    }
    mlflow.log_dict(gate_report, "gate_report.json")

### Look at the results

In [ ]:
mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment(EXPERIMENT_NAME)

runs = mlflow.search_runs(order_by=["start_time DESC"], max_results=10)

columns = [c for c in [
    "tags.mlflow.runName", "metrics.overall_score", "metrics.ingredient_usage",
    "metrics.refusal_accuracy", "metrics.rougeL", "metrics.failed_calls",
] if c in runs.columns]

runs[columns].dropna(subset=["metrics.overall_score"]) if columns else runs.head()

### Who is the Champion

In [ ]:
champion = mlflow.genai.load_prompt(f"prompts:/{grandma_personas.PROMPT_NAME}@champion")

print(f"{grandma_personas.PROMPT_NAME} @champion -> version {champion.version}")
print(f"persona: {(champion.tags or {}).get('persona')}")
print(f"variables: {champion.variables}")
print()
print(champion.template)